## Setup

In [ ]:
import os
import re
import pickle
import plotly
import numpy as np
import pandas as pd
import plotly.io as pio
from pathlib import Path
import plotly.express as px
from itertools import product
import plotly.graph_objects as go
from IPython.display import display, HTML
from src.utils.langs import LANG2S

from plotly.subplots import make_subplots

In [ ]:
figpaths = [
    "/Users/fixed/Desktop/saar/thesis/word-order-thesis/output/intervention/figures/modal_noun-adj-complete_grid.json",
    "/Users/fixed/Desktop/saar/thesis/word-order-thesis/output/intervention/figures/modal_modal-verbs_grid.json",
    "/Users/fixed/Desktop/saar/thesis/word-order-thesis/output/intervention/figures/modal_svo_grid.json"
]
for figpath in figpaths:
    fig = pio.read_json(figpath)
    pdf_path = figpath.replace(".json", ".pdf")
    fig.write_image(pdf_path, format='pdf')

In [ ]:
PATH = Path("output/intervention/probs/")
FIG_PATH = Path("output/intervention/figures/")
MODELS = ["mGPT", "aya-expanse-8b", "Meta-Llama-3-8B"]
EXPERIMENTS = ["svo", "modal-verbs", "noun-adj-complete"]

In [3]:
# def extend_token_type_into_columns(dataframe):
#     dataframe['part_of_speech'] = dataframe['token_type'].apply(lambda x: x.split('-')[0])
#     dataframe['language'] = dataframe['token_type'].apply(lambda x: x.split('-')[1])
#     dataframe['lexical_component'] = dataframe['token_type'].apply(lambda x: x.split('-')[2])

In [4]:
def get_mean_df(dataframes):
    all_dfs = []
    for df in dataframes:
        df_agg = df.groupby(['layer', 'language', 'part_of_speech', 'lexical_component'], as_index=False)['prob'].mean()
        all_dfs.append(df_agg)
    
    # Calculate mean probability for each combination across all dataframes
    mean_df = pd.concat(all_dfs).groupby(['layer', 'language', 'part_of_speech', 'lexical_component'], as_index=False)['prob'].mean()
    return mean_df

## base-plant evolution

### switch plots

In [5]:
def get_switch_heatmaps(
    df):

    # extend_token_type_into_columns(df)
    df = df.groupby(['language', 'part_of_speech', 'lexical_component', 'layer'], as_index=False)['prob'].mean()

    heatmaps = {}
    for component in ["part_of_speech", "language", "lexical_component"]:
        if component not in df.columns:
            raise ValueError(f"Expected column '{component}' not found in DataFrame")
        df_heatmap = df.pivot_table(
            index="layer",
            columns=component,
            values="prob",
            aggfunc="sum",
            fill_value=0
        )
        df_heatmap["proportion"] = df_heatmap["plant"] / (df_heatmap["plant"] + df_heatmap["base"])
        heatmaps[component] = df_heatmap
    return heatmaps

def plot_switch_heatmaps(
    heatmaps: dict,
    title=""
    ):
    # create subplots
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=tuple([k for k in heatmaps.keys()]),
        shared_xaxes=True,
        vertical_spacing=0.2
    )
    for i, (comp, heatmap) in enumerate(heatmaps.items()):
        fig.add_trace(
            go.Heatmap(
                z=[heatmap["proportion"].values],
                x=heatmap.index,
                # y=["proportion"],
                colorscale=[[0, "#ffcb30"], [1, "#a121d9"]],
                zmin=0,
                zmax=1,
                showscale=False,
            ),
            row=i+1, col=1
        )
    fig.update_layout(
        title=title,
        height=500,  # adjust height for subplots
        xaxis3_title="layer",
        yaxis_title="",
        yaxis=dict(showticklabels=False),
        yaxis2=dict(showticklabels=False),
        yaxis3=dict(showticklabels=False),
    )

    return fig

### bar plots

In [7]:
def get_combination_color_map(languages, parts_of_speech, lexical_components):
    """Pre-compute a fixed color map for all possible combinations."""
    colors = px.colors.qualitative.Safe
    combos = [
        f"$L_{{{l}}}, S_{{{p}}}, C_{{{c}}}$"
        for l, p, c in product(languages, parts_of_speech, lexical_components)
    ]
    return {combo: colors[i % len(colors)] for i, combo in enumerate(combos)}


def plot_bar_df_modal(dataframes, addition="", model_names=None, color_map=None):
    if model_names is None:
        model_names = [""] * len(dataframes)

    all_top_per_layer = []

    for dataframe, model_name in zip(dataframes, model_names):
        dataframe = dataframe.copy()
        dataframe['prob'] = pd.to_numeric(dataframe['prob'], errors='coerce')
        dataframe = dataframe.dropna(subset=['prob'])

        df_by_combo = dataframe.groupby(
            ['language', 'part_of_speech', 'lexical_component', 'layer'], as_index=False
        )['prob'].mean()

        # Top combination per layer for this dataframe
        top_per_layer = df_by_combo.loc[df_by_combo.groupby('layer')['prob'].idxmax()]
        top_per_layer = top_per_layer[['layer', 'language', 'part_of_speech', 'lexical_component', 'prob']]
        top_per_layer['model_name'] = model_name
        all_top_per_layer.append((top_per_layer, df_by_combo))

    # Across all dataframes, find the modal winning combination per layer
    all_tops = pd.concat([t for t, _ in all_top_per_layer], ignore_index=True)
    modal_combo_per_layer = (
        all_tops.groupby(['layer', 'language', 'part_of_speech', 'lexical_component'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
        .groupby('layer')
        .first()  # pick the combo with the highest count per layer
        .reset_index()
    )

    # For each layer, get the mean prob of the modal combo averaged across all dataframes
    rows = []
    for _, row in modal_combo_per_layer.iterrows():
        layer = row['layer']
        lang, pos, lex = row['language'], row['part_of_speech'], row['lexical_component']
        probs = []
        for _, df_by_combo in all_top_per_layer:
            match = df_by_combo[
                (df_by_combo['layer'] == layer) &
                (df_by_combo['language'] == lang) &
                (df_by_combo['part_of_speech'] == pos) &
                (df_by_combo['lexical_component'] == lex)
            ]
            if not match.empty:
                probs.append(match.iloc[0]['prob'])
        rows.append({
            'layer': layer,
            'prob': np.mean(probs) if probs else np.nan,
            'combination_latex': f"$L_{{{lang}}}, S_{{{pos}}}, C_{{{lex}}}$"
        })

    df_plot = pd.DataFrame(rows).sort_values('layer').reset_index(drop=True)

    fig = px.bar(
        df_plot,
        x="layer",
        y="prob",
        color="combination_latex",
        color_discrete_map=color_map,
        title=f"Most Consistently Top Combination per Layer. {addition}. (averaged across dataframes)",
        labels={"layer": "Layer", "prob": "Mean Probability", "combination_latex": "Combination"},
    )
    fig.update_layout(legend=dict(title="Token", font=dict(size=10)))
    return fig

In [8]:
color_map = get_combination_color_map(["plant", "base"], ["plant", "base"], ["plant", "base"])

In [9]:
experiment2title = {
    "noun-adj-complete": "NP",
    "svo": "SVO",
    "modal-verbs": "MV",
}

for experiment in EXPERIMENTS:
    LANGS = list(LANG2S[experiment].keys())
    all_figs = {}
    for model in MODELS:
        for src_lang in LANGS:
            datapaths = []
            for file in os.listdir(PATH):
                if file.startswith(f"{experiment}_{model}_{src_lang}") and not (file.endswith("topk.csv")):
                    datapaths.append(str(PATH / file))
            dataframes = [pd.read_csv(path) for path in datapaths]

            if dataframes:
                modal_fig = plot_bar_df_modal(dataframes, addition=f"{experiment}, {model}, {src_lang}", color_map=color_map)
                all_figs[(src_lang, model)] = modal_fig
                # switch_fig.show()
                plotly.offline.plot(modal_fig, filename = str(FIG_PATH / f"modal_{experiment}_{model}_{src_lang}.html"))
            else:
                print(f"{experiment}: No dataframes found for model {model} and source language {src_lang}. Skipping.")
    
    # Build subplot grid: rows=MODELS, cols=LANGS
    fig = make_subplots(
        rows=len(LANGS), cols=len(MODELS),
        row_titles=LANGS,
        column_titles=MODELS,
        shared_xaxes=True,
        shared_yaxes=True,
    )
    seen_legend_labels = set()

    for row_idx, src_lang in enumerate(LANGS):
        for col_idx, model in enumerate(MODELS):
            if (src_lang, model) not in all_figs:
                continue
            subfig = all_figs[(src_lang, model)]
            for trace in subfig.data:
                # Avoid duplicate legend entries
                show_legend = trace.name not in seen_legend_labels
                if show_legend:
                    seen_legend_labels.add(trace.name)
                trace.showlegend = show_legend
                fig.add_trace(trace, row=row_idx + 1, col=col_idx + 1)

    fig.update_layout(
    title=f"{experiment2title[experiment]} Dataset — Modal Combination per Layer",
    height=200 * len(LANGS),
    width=350 * len(MODELS),
    barmode="group",
    legend=dict(
        x=1.05,           # push legend outside the plot area
        y=1,
        xanchor="left",
        yanchor="top",
    ),
    margin=dict(
        r=200,            # make room on the right for the legend
        t=50,
    ),
)

    fig.show(mathjax=True)
    plotly.offline.plot(fig, filename=str(FIG_PATH / f"modal_{experiment}_grid.html"), include_plotlyjs=True, include_mathjax="cdn")
    # Instead of pickle:
    with open(FIG_PATH / f"modal_{experiment}_grid.json", "w") as f:
        f.write(pio.to_json(fig))

In [12]:
def plot_line_df(dataframe, lang_setup, model_name=""):
    # Ensure the 'prob' column is numeric
    dataframe['prob'] = pd.to_numeric(dataframe['prob'], errors='coerce')

    plotly.offline.init_notebook_mode()
    display(HTML(
        '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
    ))

    # Get the setup languages
    src_lang_base, tgt_lang_base, src_lang_plant, tgt_lang_plant = lang_setup

    # Drop rows with NaN in 'prob'
    dataframe = dataframe.dropna(subset=['prob'])
    df_agg = dataframe.groupby(['language', 'part_of_speech', 'lexical_component', 'layer'], as_index=False)['prob'].mean()
    df_agg['combination_latex'] = (
        df_agg['language'].apply(lambda x: f'L_{{{x[0]}}}') + ', ' +
        df_agg['part_of_speech'].apply(lambda x: f'S_{{{x[0]}}}') + ', ' +
        df_agg['lexical_component'].apply(lambda x: f'C_{{{x[0]}}}')
    )

    # Define custom symbol mapping
    symbol_map = {"base": "B", "plant": "P"}
    df_agg['custom_symbol'] = df_agg['part_of_speech'].map(symbol_map)

    language_colors = {
        "plant": "#1f77b4",
        "base": "#ff7f0e",
    }

    pos_text = {
        "base": "B",
        "plant": "P",
    }

    lexical_dashes = {
        "base": "solid",
        "source": "dash",
    }

    # Create a line plot for token probabilities over layers
    fig = px.line(
        df_agg,
        x="layer",
        y="prob",
        text="custom_symbol",
        color="language",
        color_discrete_map=language_colors,
        symbol="part_of_speech",
        # color=df_no_scrlang_agg['language'] + '-' + df_no_scrlang_agg['part_of_speech'],
        line_dash="lexical_component",
        title=f"Probabilities after Block Intervention (base: {src_lang_base}-{tgt_lang_base}, plant: {src_lang_plant}-{tgt_lang_plant}), model: {model_name}",
        labels={"layer": "Layer", "prob": "Probability", "combination_latex": "Token"},
    )
    fig.update_traces(showlegend=False)
    fig.update_traces(textposition='middle center')
    fig.update_traces(
        mode="lines+text",
        # text=df_no_scrlang_agg["custom_symbol"],
        textposition="middle center",
        textfont=dict(size=12),
        showlegend=False,   # we’ll build our own legend
    )

    # Add custom symbols to the plot
    for trace in fig.data:
        trace.textfont = dict(
            family="sans-serif",
            size=12,
            color=trace.line.color,
            shadow="-1px 0 black, 0 1px black, 1px 0 black, 0 -1px black",
        )
    # Update legend font size
    fig.update_layout(legend=dict(font=dict(size=14)))

    for lang, color in language_colors.items():
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="lines",
                line=dict(color=color, width=3),
                name=lang,
                legendgroup="language",
                legendgrouptitle_text="Language",
                showlegend=True,
            )
        )

    pos_legend = {
        "N (noun)": "circle",
        "A (adj)": "triangle-up",
    }

    for name, symbol in pos_legend.items():
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                marker=dict(symbol=symbol, size=0, color="#FFFFFF"),
                name=name,
                legendgroup="pos",
                legendgrouptitle_text="Part of Speech",
                showlegend=True,
            )
        )

    for lex, dash in lexical_dashes.items():
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="lines",
                line=dict(color="black", dash=dash, width=3),
                name=lex,
                legendgroup="lexical",
                legendgrouptitle_text="Lexical Component",
                showlegend=True,
            )
        )

    fig.update_layout(
        legend=dict(
            title=None,
            font=dict(size=14),
            tracegroupgap=10,  # spacing between legend sections
        )
    )

    # fig.show()
    return fig